# Análisis de Interacciones en Redes Sociales - Elecciones Presidenciales Costa Rica 2026

Este notebook realiza un análisis del crecimiento y decaimiento de las interacciones en las publicaciones de los candidatos presidenciales durante el periodo electoral.
El objetivo es responder:
1. ¿Cuántos días se demora una publicación en dejar de recibir interacciones?
2. ¿Cómo afecta el hecho de ganar las elecciones a este proceso de decaimiento?

## Visualización de Promedios Dinámicos

En este análisis, los promedios diarios se calculan utilizando **únicamente las publicaciones activas** para cada día relativo. 

1. **Interpolación Lineal**: Se utiliza para estimar valores en días intermedios donde no hubo captura de datos, asegurando una curva fluida entre el primer y último día de observación de cada post.
2. **Sin Relleno Externo**: Si una publicación deja de ser observada (ej. después del día 5), no se rellena con ceros ni se mantiene su valor anterior para días posteriores. Esto asegura que el promedio de cada día sea representativo del comportamiento real de los posts observados en ese momento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import hashlib
from scipy import stats

# Configuración de visualización estética
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.options.mode.chained_assignment = None

## Paso 1: Carga de Datos desde Excel

In [ ]:
file_path = 'Scrapping data v2.xlsx'
xl = pd.ExcelFile(file_path)

df_fb = pd.read_excel(xl, sheet_name='Facebook')
df_ig = pd.read_excel(xl, sheet_name='Instagram')
df_tk = pd.read_excel(xl, sheet_name='TikTok')
df_tw = pd.read_excel(xl, sheet_name='Twitter')
df_candidates = pd.read_excel(xl, sheet_name='Candidatos')

print("Hojas cargadas: Facebook, Instagram, TikTok, Twitter, Candidatos")

## Paso 2: Validación de Integridad y Corrección de Twitter

In [ ]:
def check_completeness(df, name, cols):
    print(f"--- Validación: {name} ---")
    df.columns = df.columns.astype(str).str.strip()
    for col in cols:
        if col in df.columns:
            nulls = df[col].isnull().sum()
            print(f"Nulos en '{col}': {nulls}")
        else:
            print(f"¡ADVERTENCIA!: La columna '{col}' no existe en {name}")

check_completeness(df_fb, "Facebook", ['postId', 'url', 'texto'])
check_completeness(df_ig, "Instagram", ['postId', 'url', 'texto'])
check_completeness(df_tk, "TikTok", ['postId', 'url', 'texto'])
check_completeness(df_tw, "Twitter", ['postId', 'url', 'text'])

def generate_id(text):
    if pd.isna(text): return "missing_text"
    return hashlib.md5(str(text).encode('utf-8')).hexdigest()

if df_tw['postId'].isnull().all() or df_tw['postId'].astype(str).str.strip().eq('').all():
    print("\nGenerando IDs únicos para Twitter basados en el contenido del texto...")
    df_tw['postId'] = df_tw['text'].apply(generate_id)
    df_tw['url'] = "https://twitter.com/x/status/" + df_tw['postId']

print(f"\nTwitter - Registros únicos por postId: {df_tw['postId'].nunique()}")

## Paso 3: Preparación de la Evolución Temporal

In [ ]:
def prepare_evolution_df(df, platform, mapping, user_col, id_col, date_pub_col):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    
    # Identificar columna de candidato
    actual_user_col = next((col for col in [user_col, 'Username', 'name', 'Candidato'] if col in df.columns), None)
    if actual_user_col:
        df['candidate'] = df[actual_user_col].astype(str)
    else:
        df['candidate'] = "Desconocido"
    
    # Filtrar estrictamente candidatos válidos (excluir Desconocido)
    df = df[~df['candidate'].str.contains('Desconocido|nan|None', case=False, na=True)]
    
    df['platform'] = platform
    df['internal_post_id'] = platform + "_" + df[id_col].astype(str)
    df = df.rename(columns=mapping)
    
    df['fecha_ext'] = pd.to_datetime(df['fecha_ext'], errors='coerce', dayfirst=True).dt.tz_localize(None)
    df['fecha_pub'] = pd.to_datetime(df[date_pub_col], errors='coerce', dayfirst=True).dt.tz_localize(None)
    df['dia_relativo'] = (df['fecha_ext'] - df['fecha_pub']).dt.days + 1
    
    for col in ['likes', 'comments', 'shares']:
        if col in df.columns: 
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else: 
            df[col] = 0.0
            
    return df[['internal_post_id', 'platform', 'candidate', 'dia_relativo', 'likes', 'comments', 'shares']]

ev_fb = prepare_evolution_df(df_fb, 'Facebook', {'megusta': 'likes', 'comentarios': 'comments', 'compartidas': 'shares'}, 'Username', 'postId', 'fecha_pub')
ev_ig = prepare_evolution_df(df_ig, 'Instagram', {'megusta': 'likes', 'comentarios': 'comments'}, 'Username', 'postId', 'fecha_pub')
ev_tk = prepare_evolution_df(df_tk, 'TikTok', {'megusta': 'likes', 'comentarios': 'comments', 'compartidos': 'shares'}, 'Username', 'postId', 'fecha_pub')
ev_tw = prepare_evolution_df(df_tw, 'Twitter', {'likecount': 'likes', 'replycount': 'comments', 'retweet_count': 'shares'}, 'Username', 'postId', 'createdat')

df_combined = pd.concat([ev_fb, ev_ig, ev_tk, ev_tw], ignore_index=True)

## Paso 4: Interpolación y Estabilización

Implementamos interpolación lineal para días faltantes dentro del rango observado de cada post.

In [ ]:
def interpolate_and_calc_pct(group):
    group = group.sort_values('dia_relativo').drop_duplicates('dia_relativo')
    if len(group) < 1: return None
    
    # Asegurar crecimiento monótono
    group[['likes', 'comments', 'shares']] = group[['likes', 'comments', 'shares']].cummax()
    
    min_day, max_day = int(group['dia_relativo'].min()), int(group['dia_relativo'].max())
    
    if max_day > min_day:
        full_range = pd.DataFrame({'dia_relativo': range(min_day, max_day + 1)})
        meta = group.iloc[0][['internal_post_id', 'platform', 'candidate']]
        group = pd.merge(full_range, group, on='dia_relativo', how='left')
        for col in ['internal_post_id', 'platform', 'candidate']: 
            group[col] = meta[col]
        group[['likes', 'comments', 'shares']] = group[['likes', 'comments', 'shares']].interpolate(method='linear')
    
    for col in ['likes', 'comments', 'shares']:
        total = group[col].max()
        group[f'{col}_pct'] = (group[col] / total * 100) if total > 0 else 0
        
    return group

df_pct = df_combined.groupby('internal_post_id', group_keys=False).apply(interpolate_and_calc_pct)
df_pct = df_pct.dropna(subset=['internal_post_id'])
print("Interpolación terminada. Solo se incluyen posts activos para cada día en los promedios.")

## Paso 5: Visualización de Curvas de Interacción

In [ ]:
def plot_all_curves():
    candidates = sorted(df_pct['candidate'].unique())
    platforms = df_pct['platform'].unique()
    
    for candidate in candidates:
        for platform in platforms:
            subset = df_pct[(df_pct['candidate'] == candidate) & (df_pct['platform'] == platform)]
            if subset.empty: continue
            
            # Promedio de publicaciones activas en cada día
            avg_evol = subset.groupby('dia_relativo')[['likes_pct', 'comments_pct', 'shares_pct']].mean().reset_index()
            
            zero_pt = pd.DataFrame({'dia_relativo': [0], 'likes_pct': [0.0], 'comments_pct': [0.0], 'shares_pct': [0.0]})
            avg_evol = pd.concat([zero_pt, avg_evol]).sort_values('dia_relativo')
            
            plt.figure(figsize=(9, 4.5))
            plt.plot(avg_evol['dia_relativo'], avg_evol['likes_pct'], label='Likes %', marker='o', markersize=4)
            plt.plot(avg_evol['dia_relativo'], avg_evol['comments_pct'], label='Comentarios %', marker='s', markersize=4)
            plt.plot(avg_evol['dia_relativo'], avg_evol['shares_pct'], label='Compartidos %', marker='^', markersize=4)
            plt.title(f'Evolución: {candidate} en {platform}')
            plt.xlabel('Día Relativo')
            plt.ylabel('% Acumulado Promedio')
            plt.ylim(-5, 105)
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

plot_all_curves()

## Paso 6: Análisis Estadístico de Significancia

In [ ]:
df_pct['likes_growth'] = df_pct.groupby('internal_post_id')['likes_pct'].diff().fillna(0)
df_pct['is_winner'] = df_pct['candidate'].str.contains('Laura', na=False)

print("Análisis de Significancia Estadística del Crecimiento (Día a Día):")
for d in range(1, 11):
    growth_day = df_pct[df_pct['dia_relativo'] == d]['likes_growth']
    if len(growth_day) > 1:
        t_stat, p_val = stats.ttest_1samp(growth_day, 0)
        status = "ACTIVO" if p_val < 0.05 else "MUERTO/ESTÁTICO"
        print(f"Día {d}: Crecimiento prom {growth_day.mean():.2f}% | Status: {status} (p={p_val:.4f})")

plt.figure(figsize=(10, 5))
winner_subset = df_pct[df_pct['is_winner']]
others_subset = df_pct[~df_pct['is_winner']]

winner_avg = winner_subset.groupby('dia_relativo')['likes_pct'].mean().reset_index()
others_avg = others_subset.groupby('dia_relativo')['likes_pct'].mean().reset_index()

plt.plot(winner_avg['dia_relativo'], winner_avg['likes_pct'], label='Laura Fernández (Ganadora)', color='gold', linewidth=4, marker='*')
plt.plot(others_avg['dia_relativo'], others_avg['likes_pct'], label='Resto de Candidatos', color='darkblue', linestyle='--', marker='o')
plt.title('Persistencia de Atención: Ganadora vs Perdedores')
plt.xlabel('Día Relativo')
plt.ylabel('% de Interacciones Totales Promedio')
plt.legend()
plt.show()

## Conclusiones

1. **Ventana de Engagement**: El crecimiento se detiene significativamente después de los primeros días.
2. **Dinámica de Ganador**: Comparativa visual entre candidatos.